# Treinamento do Classificador de Sentimento (IMDB) - TextCNN

Treina uma rede neural real (TextCNN, PyTorch) do zero no [Large Movie Review Dataset (IMDB)](https://ai.stanford.edu/~amaas/data/sentiment/), e depois **congela** a rede treinada em um grafo estatico (TorchScript) para servir em produção - o mesmo conceito do `export(...)` para MindIR + `nn.GraphCell` usado no material de Cloud Side Inference (MindSpore).

Saida deste notebook, salva neste mesmo diretorio (lida por `app/backend/routers/classifier.py`):
- `model_frozen.pt` - o grafo congelado (TorchScript), sem depender da classe `TextCNN` para ser carregado.
- `vocab.json` - vocabulario palavra -> id usado no pre-processamento.
- `metadata.json` - versao, dataset, acuracia de teste, hiperparametros.

Baseado no TextCNN treinado em `HuaweiLabs/HCIA/DeepLearning-&-Framework/Framework/TextCNN.ipynb`.

In [1]:
import json
import os
import re
import tarfile
import urllib.request
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(42)

MODEL_DIR = Path.cwd()
DATA_DIR = MODEL_DIR / "data"
DATASET_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
ARCHIVE_PATH = DATA_DIR / "aclImdb_v1.tar.gz"
EXTRACTED_DIR = DATA_DIR / "aclImdb"

MODEL_DIR, DATA_DIR

(PosixPath('/home/joao/Documentos/iftech/app/backend/ml/model'),
 PosixPath('/home/joao/Documentos/iftech/app/backend/ml/model/data'))

## 1. Dataset

Baixa e extrai o dataset (~84MB) apenas na primeira execucao.

In [2]:
def download_imdb(data_dir):
    data_dir.mkdir(exist_ok=True)

    if not EXTRACTED_DIR.exists():
        if not ARCHIVE_PATH.exists():
            print(f"Baixando dataset de {DATASET_URL} ...")
            urllib.request.urlretrieve(DATASET_URL, ARCHIVE_PATH)
        print("Extraindo...")
        with tarfile.open(ARCHIVE_PATH) as tar:
            tar.extractall(data_dir, filter="data")
    else:
        print("Dataset ja extraido, pulando download.")


download_imdb(DATA_DIR)

Dataset ja extraido, pulando download.


In [3]:
def load_imdb_data(extracted_dir):
    reviews = []
    labels = []
    for label_type in ["pos", "neg"]:
        dir_name = extracted_dir / "train" / label_type
        for file_path in sorted(dir_name.glob("*.txt")):
            reviews.append(file_path.read_text(encoding="utf-8"))
            labels.append(1 if label_type == "pos" else 0)
    return reviews, labels

## 2. Pre-processamento e vocabulario

In [4]:
def preprocess_text(text):
    text = re.sub(r"[^\w\s]", "", text)
    text = text.lower()
    return text


def build_vocab(reviews, max_vocab_size=10000):
    words_count = Counter()
    for review in reviews:
        words_count.update(review.split())
    vocab = {word: i + 1 for i, (word, _) in enumerate(words_count.most_common(max_vocab_size))}
    vocab["<PAD>"] = 0
    return vocab


def text_to_sequence(text, vocab):
    return [vocab.get(word, 0) for word in text.split()]


def pad_sequences(seq, max_len):
    if len(seq) >= max_len:
        return seq[:max_len]
    return seq + [0] * (max_len - len(seq))

## 3. Dataset PyTorch

In [5]:
class IMDBDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

## 4. Modelo TextCNN

In [6]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, kernel_sizes=(3, 4, 5), num_filters=100):
        super(TextCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.convs = nn.ModuleList([
            nn.Conv2d(1, num_filters, (k, embed_dim)) for k in kernel_sizes
        ])
        self.fc = nn.Linear(len(kernel_sizes) * num_filters, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.embedding(x)
        x = x.unsqueeze(1)
        x = [torch.relu(conv(x)).squeeze(3) for conv in self.convs]
        x = [torch.max_pool1d(i, i.size(2)).squeeze(2) for i in x]
        x = torch.cat(x, 1)
        x = self.dropout(x)
        return self.fc(x)

## 5. Treino e avaliacao

In [7]:
def train_model(model, train_loader, criterion, optimizer, device, num_epochs):
    model.to(device)
    model.train()

    for epoch in range(num_epochs):
        total_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {total_loss / len(train_loader):.4f}")


def test_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
    accuracy = correct / total
    print(f"Acuracia no teste: {100 * accuracy:.2f}%")
    return accuracy

## 6. Programa principal

In [8]:
reviews, labels = load_imdb_data(EXTRACTED_DIR)
reviews = [preprocess_text(review) for review in reviews]

vocab = build_vocab(reviews)
max_len = 200

sequences = [text_to_sequence(review, vocab) for review in reviews]
X = torch.tensor([pad_sequences(seq, max_len) for seq in sequences], dtype=torch.long)
y = torch.tensor(labels, dtype=torch.long)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_loader = DataLoader(IMDBDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(IMDBDataset(X_test, y_test), batch_size=64, shuffle=False)

vocab_size = len(vocab) + 10
embed_dim = 100
num_classes = 2
num_epochs = 20

model = TextCNN(vocab_size, embed_dim, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando:", device)

train_model(model, train_loader, criterion, optimizer, device, num_epochs)
test_accuracy = test_model(model, test_loader, device)

Usando: cpu


Epoch [1/20], Loss: 0.6753


Epoch [2/20], Loss: 0.5245


Epoch [3/20], Loss: 0.4648


Epoch [4/20], Loss: 0.3919


...

Epoch [17/20], Loss: 0.0342


Epoch [18/20], Loss: 0.0320


Epoch [19/20], Loss: 0.0317


Epoch [20/20], Loss: 0.0284


Acuracia no teste: 84.18%


## 7. Congelamento do modelo (TorchScript)

Traca o grafo computacional do modelo treinado com um tensor de exemplo e salva o resultado como um artefato autocontido (`model_frozen.pt`) - o servico de inferencia carrega esse arquivo sem precisar da classe `TextCNN`, da mesma forma que `ms.load('mymodel.mindir')` + `nn.GraphCell` carregam um MindIR sem a definicao original da rede.

In [9]:
model.eval()
example_input = torch.zeros((1, max_len), dtype=torch.long)
traced_model = torch.jit.trace(model, example_input)

frozen_path = MODEL_DIR / "model_frozen.pt"
traced_model.save(str(frozen_path))
print(f"Modelo congelado salvo em {frozen_path}")

Modelo congelado salvo em /home/joao/Documentos/iftech/app/backend/ml/model/model_frozen.pt


## 8. Vocabulario e metadata (versionamento)

In [10]:
VERSION = "v1"

(MODEL_DIR / "vocab.json").write_text(json.dumps(vocab, ensure_ascii=False))

metadata = {
    "version": VERSION,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "framework": "pytorch+torchscript",
    "architecture": "TextCNN",
    "dataset": "aclImdb (Stanford Large Movie Review Dataset)",
    "vocab_size": vocab_size,
    "embed_dim": embed_dim,
    "max_len": max_len,
    "num_epochs": num_epochs,
    "n_train": len(X_train),
    "n_test": len(X_test),
    "test_accuracy": round(float(test_accuracy), 4),
}
(MODEL_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False))

print(f"Metadata salva - versao {VERSION}, acuracia de teste {test_accuracy:.2%}")
metadata

Metadata salva - versao v1, acuracia de teste 84.18%


{'version': 'v1',
 'trained_at': '2026-07-21T13:28:18.549849+00:00',
 'framework': 'pytorch+torchscript',
 'architecture': 'TextCNN',
 'dataset': 'aclImdb (Stanford Large Movie Review Dataset)',
 'vocab_size': 10011,
 'embed_dim': 100,
 'max_len': 200,
 'num_epochs': 20,
 'n_train': 20000,
 'n_test': 5000,
 'test_accuracy': 0.8418}

## 9. Sanity check - carregando o artefato congelado do zero

Recarrega `model_frozen.pt` sem usar a classe `TextCNN` (simulando o que o backend faz).

In [11]:
reloaded_model = torch.jit.load(str(frozen_path))
reloaded_model.eval()

reloaded_vocab = json.loads((MODEL_DIR / "vocab.json").read_text())


def predict_sentiment(text, jit_model, vocab_map, max_len):
    sequence = pad_sequences(text_to_sequence(preprocess_text(text), vocab_map), max_len)
    input_tensor = torch.tensor(sequence, dtype=torch.long).unsqueeze(0)
    with torch.no_grad():
        logits = jit_model(input_tensor)
        probs = torch.softmax(logits, dim=1)[0]
    predicted_class = int(probs.argmax())
    label = "Positivo" if predicted_class == 1 else "Negativo"
    return label, float(probs[predicted_class])


samples = [
    "This movie was absolutely wonderful, the acting was great and I loved every minute.",
    "Terrible film, a complete waste of time. Boring and poorly acted.",
]

for text in samples:
    label, confidence = predict_sentiment(text, reloaded_model, reloaded_vocab, max_len)
    print(f"{label} (confianca {confidence:.2%}) -> {text[:60]}...")

Positivo (confianca 99.98%) -> This movie was absolutely wonderful, the acting was great an...
Negativo (confianca 100.00%) -> Terrible film, a complete waste of time. Boring and poorly a...
